<a href="https://colab.research.google.com/github/notslok/Computer-Networks-Notes/blob/master/pcapng_to_ndjson.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PCAP → NDJSON Converter for Bus Inspector

Reads `merged.pcapng` and writes `traffic.ndjson` ready for the bus-inspector visualizer.

---

### What gets extracted

| # | Protocol | Interface | Participants | Description |
|---|---|---|---|---|
| 1 | **AVTP / IEEE-1722** | iface 1 (Ethernet) | RPI ↔ Zone2 | AVTP-encapsulated CAN frames on the Ethernet bus |
| 2 | **ETAS BOA TCP:22001** | iface 0 (ES886 tap) | ES886 → PC | Raw CAN frames tapped from physical CAN bus and forwarded over TCP |

### Output event schema
```json
{ "ts": 1772622657.619,
  "source": "d8:3a:dd:2a:27:c7",  "target": "00:03:19:00:00:02",
  "proto": "AVTP-CAN",
  "can_id": "0x020",  "bus_id": 1,  "dlc": 4,
  "can_data": "0000e040",  "bytes": 56,  "iface": 1 }
```

### No external dependencies — pure stdlib only

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  CONFIG  — edit these paths before running
# ─────────────────────────────────────────────────────────────────────────────

PCAP_PATH   = "merged.pcapng"   # input .pcapng file
OUTPUT_PATH = "traffic.ndjson"  # output — drop this into bus-inspector/public/

# Set to an integer to limit output (useful for testing), or None for all
MAX_EVENTS  = None

In [ ]:
import struct, json, collections
from pathlib import Path

data = Path(PCAP_PATH).read_bytes()
print(f"Loaded {PCAP_PATH}  —  {len(data):,} bytes")

## 1 — pcapng block parser

In [ ]:
def parse_pcapng(data: bytes):
    """
    Walk pcapng blocks and yield (iface_id, ts_seconds, packet_bytes)
    for every Enhanced Packet Block (type 0x00000006).

    Handles the if_tsresol IDB option so timestamps are always in seconds
    regardless of whether the capture used micro- or nano-second resolution.
    """
    pos          = 0
    ts_res: dict = {}          # iface_id -> ticks-per-second
    idb_index    = 0

    while pos < len(data) - 8:
        try:
            block_type = struct.unpack_from("<I", data, pos)[0]
            block_len  = struct.unpack_from("<I", data, pos + 4)[0]

            if block_len < 12 or block_len > 0x10000000 or pos + block_len > len(data):
                pos += 4
                continue

            # ── Interface Description Block (IDB) ──────────────────────────
            if block_type == 1:
                resolution = 1_000_000  # default: microseconds

                # Walk options inside the IDB
                opt_pos = pos + 12      # skip block_type(4)+len(4)+link_type(2)+reserved(2)
                while opt_pos + 4 <= pos + block_len - 4:
                    opt_code = struct.unpack_from("<H", data, opt_pos)[0]
                    opt_len  = struct.unpack_from("<H", data, opt_pos + 2)[0]
                    if opt_code == 9 and opt_len >= 1:   # if_tsresol
                        b = data[opt_pos + 4]
                        resolution = (2 if (b & 0x80) else 10) ** (b & 0x7f)
                    if opt_code == 0:                    # opt_endofopt
                        break
                    opt_pos += 4 + opt_len + ((-opt_len) % 4)

                ts_res[idb_index] = resolution
                idb_index += 1

            # ── Enhanced Packet Block (EPB) ────────────────────────────────
            elif block_type == 6:
                iface_id = struct.unpack_from("<I", data, pos + 8)[0]
                ts_hi    = struct.unpack_from("<I", data, pos + 12)[0]
                ts_lo    = struct.unpack_from("<I", data, pos + 16)[0]
                cap_len  = struct.unpack_from("<I", data, pos + 20)[0]
                ticks    = (ts_hi << 32) | ts_lo
                ts_sec   = ticks / ts_res.get(iface_id, 1_000_000)
                pkt      = data[pos + 28 : pos + 28 + cap_len]
                yield iface_id, ts_sec, pkt

            pos += block_len

        except Exception:
            pos += 4


# Sanity check
total_pkts = sum(1 for _ in parse_pcapng(data))
print(f"Total packets in file: {total_pkts:,}")

## 2 — Protocol decoders

### 2a — AVTP / IEEE-1722 NTSCF + ACF CAN Brief

```
Ethernet frame layout (relevant bytes):
  [0..5]   dst MAC
  [6..11]  src MAC
  [12..13] EtherType = 0x22f0
  [14]     AVTP subtype = 0x82  (NTSCF)
  [17]     sequence number
  [18..25] stream_id (8 bytes)
  [26..27] ACF header word:  msg_type[15:9]  |  msg_len_quadlets[8:0]
  [28]     ACF flags
  [29]     bus_id  (5 LSBs)
  [32..33] CAN ID  (11 LSBs = standard 11-bit CAN ID)
  [34..]   CAN data  (acf_len_quadlets*4 - 8 bytes)
```

In [ ]:
def decode_avtp_can(pkt: bytes) -> dict | None:
    if len(pkt) < 35:
        return None
    if struct.unpack_from(">H", pkt, 12)[0] != 0x22f0:   # EtherType IEEE-1722
        return None
    if pkt[14] != 0x82:                                   # NTSCF subtype
        return None

    acf_word  = struct.unpack_from(">H", pkt, 26)[0]
    acf_type  = (acf_word >> 9) & 0x7f
    if acf_type != 2:                                     # ACF CAN Brief = 2
        return None

    acf_len_q = acf_word & 0x01ff
    data_len  = acf_len_q * 4 - 8                         # subtract 8-byte ACF header
    can_id    = struct.unpack_from(">H", pkt, 32)[0] & 0x07ff
    can_data  = pkt[34 : 34 + data_len].hex() if len(pkt) >= 34 + data_len else ""

    return {
        "proto":    "AVTP-CAN",
        "source":   ":".join(f"{b:02x}" for b in pkt[6:12]),
        "target":   ":".join(f"{b:02x}" for b in pkt[0:6]),
        "seq":      pkt[17],
        "bus_id":   pkt[29] & 0x1f,
        "can_id":   f"0x{can_id:03x}",
        "dlc":      data_len,
        "can_data": can_data,
        "bytes":    len(pkt),
    }

### 2b — ETAS BOA EDE_CAN_DATA  (TCP port 22001, from ES886)

```
TCP payload layout:
  [0]      source_id  (1 byte)
  [1..3]   TLV type   (3 bytes BE) — CAN RX/TX Data = 0x000001
  [4..7]   length     (4 bytes BE)
  ── CAN block (offset 8 into TCP payload) ──
  [8..11]  reserved / padding
  [12..15] hardware timestamp  (4 bytes BE)
  [16..19] flags + DLC packed  (upper 28 bits = flags, lower 4 bits = DLC)
  [20..23] frame_id            (4 bytes BE) — CAN ID = (frame_id >> 18) & 0x7ff
  [24..27] tag
  [28..]   CAN data            (DLC bytes)
```

In [ ]:
def decode_boa_can(pkt: bytes) -> dict | None:
    if len(pkt) < 14:
        return None
    if struct.unpack_from(">H", pkt, 12)[0] != 0x0800:   # IPv4
        return None
    if len(pkt) < 34 or pkt[23] != 6:                    # TCP protocol
        return None

    ihl     = (pkt[14] & 0x0f) * 4
    tcp_off = 14 + ihl
    if len(pkt) < tcp_off + 20:
        return None

    sport = struct.unpack_from(">H", pkt, tcp_off)[0]
    dport = struct.unpack_from(">H", pkt, tcp_off + 2)[0]
    if sport != 22001 and dport != 22001:
        return None

    tcp_hlen = (pkt[tcp_off + 12] >> 4) * 4
    payload  = pkt[tcp_off + tcp_hlen:]
    if len(payload) < 28:
        return None

    tlv = struct.unpack_from(">I", payload, 0)[0] & 0x00ffffff
    if tlv != 1:                                          # CAN RX/TX Data TLV
        return None

    can = payload[8:]                                     # CAN block starts here
    if len(can) < 20:
        return None

    ts_hw     = struct.unpack_from(">I", can, 4)[0]
    flags_dlc = struct.unpack_from(">I", can, 8)[0]
    dlc       = flags_dlc & 0xf
    frame_id  = struct.unpack_from(">I", can, 12)[0]
    can_id    = (frame_id >> 18) & 0x7ff
    can_data  = can[20 : 20 + dlc].hex() if len(can) >= 20 + dlc else ""

    return {
        "proto":    "BOA-CAN",
        "source":   ":".join(f"{b:02x}" for b in pkt[6:12]),
        "target":   ":".join(f"{b:02x}" for b in pkt[0:6]),
        "src_ip":   ".".join(str(b) for b in pkt[26:30]),
        "dst_ip":   ".".join(str(b) for b in pkt[30:34]),
        "ts_hw":    ts_hw,
        "can_id":   f"0x{can_id:03x}",
        "dlc":      dlc,
        "can_data": can_data,
        "frame_id": f"0x{frame_id:08x}",
        "bytes":    len(pkt),
    }

## 3 — Decode all packets

In [ ]:
events       = []
skipped      = 0
proto_counts = collections.Counter()

for iface_id, ts_sec, pkt in parse_pcapng(data):
    event = decode_avtp_can(pkt) or decode_boa_can(pkt)

    if event is None:
        skipped += 1
        continue

    event["ts"]    = round(ts_sec, 6)
    event["iface"] = iface_id
    proto_counts[event["proto"]] += 1
    events.append(event)

    if MAX_EVENTS and len(events) >= MAX_EVENTS:
        break

print(f"Decoded : {len(events):,} events")
print(f"Skipped : {skipped:,} packets (ARP, mDNS, keepalives, etc.)")
print()
print("Protocol breakdown:")
for proto, count in proto_counts.most_common():
    print(f"  {proto:<12}  {count:>8,}")

## 4 — Stats

In [ ]:
can_id_counts  = collections.Counter(e["can_id"]  for e in events)
source_counts  = collections.Counter(e["source"]  for e in events)
target_counts  = collections.Counter(e["target"]  for e in events)

print("CAN IDs observed:")
for cid, cnt in can_id_counts.most_common():
    print(f"  {cid}  →  {cnt:,}")

print("\nSource MACs:")
for mac, cnt in source_counts.most_common():
    print(f"  {mac}  →  {cnt:,}")

print("\nDestination MACs / IPs:")
for addr, cnt in target_counts.most_common():
    print(f"  {addr}  →  {cnt:,}")

if events:
    ts_start = min(e["ts"] for e in events)
    ts_end   = max(e["ts"] for e in events)
    duration = ts_end - ts_start
    print(f"\nCapture duration : {duration:.2f} s")
    print(f"Average rate     : {len(events)/duration:.0f} events/s")

## 5 — Preview

In [ ]:
print("First 6 decoded events:\n")
for i, e in enumerate(events[:6]):
    print(f"[{i}] {json.dumps(e)}")

## 6 — (Optional) Filter

Edit the cell below to narrow what gets written to the ndjson.

```python
# Examples:
filtered = [e for e in events if e['proto'] == 'AVTP-CAN']         # Ethernet bus only
filtered = [e for e in events if e['proto'] == 'BOA-CAN']          # CAN tap only
filtered = [e for e in events if e['can_id'] == '0x020']           # single CAN ID
filtered = [e for e in events if e['source'] == 'd8:3a:dd:2a:27:c7']  # RPI traffic only
filtered = events[::3]                                              # every 3rd event (rate reduction)
```

In [ ]:
filtered = events     # ← change this line to apply a filter

print(f"Events after filter: {len(filtered):,}  (of {len(events):,} total)")

## 7 — Write ndjson

In [ ]:
out = Path(OUTPUT_PATH)
with out.open("w") as f:
    for event in filtered:
        f.write(json.dumps(event, separators=(",", ":")) + "\n")

size_kb = out.stat().st_size / 1024
print(f"✓ Wrote {len(filtered):,} lines  →  {out}  ({size_kb:.0f} KB)")
print()
print("Next step: copy traffic.ndjson into bus-inspector/public/ and click Start Traffic.")

## 8 — Verify output

In [ ]:
lines = Path(OUTPUT_PATH).read_text().splitlines()

print(f"Lines in file: {len(lines):,}\n")

def fmt(line):
    o = json.loads(line)
    return (f"  proto={o['proto']:<10}  src={o['source']}  "
            f"can_id={o['can_id']}  dlc={o['dlc']}  data={o['can_data']}")

print("First 3:")
for l in lines[:3]:  print(fmt(l))

print("\nLast 3:")
for l in lines[-3:]: print(fmt(l))